# Prompt Generator Testing

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))


from agents.prompt_generator_agent import PromptGeneratorAgent
from agents.schema_generator_agent import SchemaGeneratorAgent
from spectrum_client import *

## Mock Unit Test

In [2]:
from unittest.mock import patch

prompt_generator_agent = PromptGeneratorAgent(model='gpt-4.1-mini')

def fake_response(text):
    return {"output": [{"type": "message", "content": [{"type": "text", "text": text}]}]}

intent_response = fake_response('{"intent": "generate"}')
generate_response = fake_response('{"system_prompt": "...", "user_prompt": "{{TRANSCRIPT}}", "metadata_fields": null, "output_format": {"sentiment": "Literal[\'positive\',\'negative\',\'neutral\'] — sentiment of the call"}}')

with patch.object(prompt_generator_agent, '_post', side_effect=[intent_response, generate_response]):
    result = prompt_generator_agent.run(user_input="create a prompt...")
    print(result)
    print(prompt_generator_agent.messages)
    print(prompt_generator_agent.reset())
    print(prompt_generator_agent.messages)


system_prompt='...' user_prompt='{{TRANSCRIPT}}' metadata_fields=None output_format={'sentiment': "Literal['positive','negative','neutral'] — sentiment of the call"}
[{'role': 'system', 'content': '\nYou are an expert prompt engineer tasked with creating well-designed prompts to help extract and analyze data from transcripts between call center agents and customers.\n\nGiven a plain-language request, produce a Jinja2 prompt template with the following structure:\n{\n    "system_prompt": str,        # task instructions for the LLM\n    "user_prompt": str,          # context + transcript using {{TRANSCRIPT}} as the injection point\n    "output_format": dict,       # REQUIRED — a JSON object where each key is a field name and each value is "<type> — <description>"\n    "metadata_fields": list[str] # optional — any metadata fields to inject into the prompt\n}\n\nRules:\n- output_format is MANDATORY. Every field the prompt asks the LLM to extract must appear here with its name, type, and a 

## Unit Test - Actual API Call

In [21]:
test_prompt_1 = """
I need a prompt to identify the top 15 call drivers by category but i dont know the initial categories so we need a prompt to do some discovery for the categories from transcripts. So, I need this to look at a transcript and identify the main reason for calling, the call reason category, the ai reasoining, specific transcript quote
"""

In [22]:
prompt_generator_agent = PromptGeneratorAgent(model='gpt-4.1-mini')

result = prompt_generator_agent.run(user_input=test_prompt_1)

In [23]:
print(result.system_prompt)

You are a data analyst AI tasked with analyzing call center transcripts to discover and categorize the main reasons customers call. Your goal is to identify distinct call reason categories from the transcript, determine the primary reason for the call, provide AI reasoning for the categorization, and extract a specific quote from the transcript that best illustrates the call reason.


In [24]:
print(result.user_prompt)

Analyze the following call center transcript to identify the main reason for the customer's call. Since the initial categories are unknown, discover and define the call reason category based on the transcript content. Provide your AI reasoning for assigning this category and extract a specific quote from the transcript that best supports your conclusion.

Transcript:
{{TRANSCRIPT}}

Respond ONLY in the following JSON format:
{
    "main_call_reason": "string — the primary reason the customer is calling",
    "call_reason_category": "string — the discovered category that best describes the call reason",
    "ai_reasoning": "string — explanation of how the category was determined based on the transcript",
    "supporting_quote": "string — a specific excerpt from the transcript that illustrates the call reason"
}


In [25]:
result_2 = prompt_generator_agent.run(user_input="Does this use a pydantic response format?")

In [26]:
print(result_2)

{
  "system_prompt": "You are a data analyst AI tasked with analyzing call center transcripts to discover and categorize the main reasons customers call. Your goal is to identify distinct call reason categories from the transcript, determine the primary reason for the call, provide AI reasoning for the categorization, and extract a specific quote from the transcript that best illustrates the call reason.",
  "user_prompt": "Analyze the following call center transcript to identify the main reason for the customer's call. Since the initial categories are unknown, discover and define the call reason category based on the transcript content. Provide your AI reasoning for assigning this category and extract a specific quote from the transcript that best supports your conclusion.\n\nTranscript:\n{{TRANSCRIPT}}\n\nRespond ONLY in the following JSON format:\n{\n    \"main_call_reason\": \"string — the primary reason the customer is calling\",\n    \"call_reason_category\": \"string — the disco

In [27]:
prompt_generator_agent.reset() # clear chat hsitory

# Schema Genarator Testing

## Mock Unit Test

In [28]:
from unittest.mock import patch
from agents.schema_generator_agent import SchemaGeneratorAgent
from data_models.agent_models.prompt_generator_prompt_model import PromptModel

agent = SchemaGeneratorAgent(model='gpt-4.1-mini')

prompt_model = PromptModel(
    system_prompt="...",
    user_prompt="...",
    metadata_fields=None,
    output_format={"sentiment": "Literal['positive','negative','neutral'] — sentiment of the call"}
)

def fake_response(text):
    return {"output": [{"type": "message", "content": [{"type": "text", "text": text}]}]}

valid_code = """
from pydantic import BaseModel, Field
from typing import Literal

class SentimentResult(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(..., description="sentiment of the call")
"""

schema_response = fake_response(json.dumps({
    "model_name": "SentimentResult",
    "code": valid_code,
    "prompt_feedback": ""
}))

with patch.object(agent, '_post', return_value=schema_response):
    test_result, path = agent.run(prompt_model, "analyse sentiment", output_dir="data_models/generated")


assert path.exists()


In [29]:
bad_code = 'from pydantic import BaseModel\nclass SentimentResult(BaseModel):\n    sentiment: UndefinedType'
bad_schema_response = fake_response('{"model_name": "SentimentResult", "code": "' + bad_code.replace('\n', '\\n') + '", "prompt_feedback": ""}')
good_schema_response = fake_response('{"model_name": "SentimentResult", "code": "' + valid_code.replace('\n', '\\n') + '", "prompt_feedback": ""}')

# call 1: LLM returns bad code → write_code raises → call 2: LLM returns fixed code
with patch.object(agent, '_post', side_effect=[bad_schema_response, good_schema_response]):
    test_result, path = agent.run(prompt_model, "analyse sentiment", output_dir="data_models/generated")


[SchemaGeneratorAgent] Import failed (attempt 1): name 'UndefinedType' is not defined. Asking LLM to fix...


ValidationError: 2 validation errors for SchemaGeneratorResult
model_name
  Field required [type=missing, input_value={'{"model_name"': '"Senti..."prompt_feedback": ""}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
code
  Field required [type=missing, input_value={'{"model_name"': '"Senti..."prompt_feedback": ""}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

## Unit Test - Actual API Call

In [ ]:
schema_agent = SchemaGeneratorAgent(model='gpt-4.1-mini')
schema_result, path = schema_agent.run(
    prompt_model=result,
    user_input=result.system_prompt,
    output_dir="data_models/generated"
)
# result.code, result.model_name, result.prompt_feedback
# path → data_models/generated/sentiment_analysis_result.py

# process_batch + TranscriptChunker Testing

## Example — extract_chunks and format_chunks_for_prompt

In [ ]:
from tools.transcript_chunker import extract_chunks, format_chunks_for_prompt

SAMPLE_TRANSCRIPT = """Agent: Thank you for calling, how can I help you today?
Customer: Hi, I want to cancel my subscription. I've been charged twice this month.
Agent: I'm sorry to hear that. Can I get your account number?
Customer: Sure, it's 12345. The double charge is really frustrating.
Agent: I can see the duplicate charge. I'll process a refund right away.
Customer: Thank you, I appreciate that.
Agent: Is there anything else I can help you with?
Customer: No, that's all. Goodbye."""

# Simulated extraction output — what the LLM would return for this transcript
extraction_output = {
    "call_reason": "billing dispute",
    "sentiment": "negative",
    "quote": "I've been charged twice this month",
    "reasoning": "Customer explicitly mentions a double charge and expresses frustration",
}

chunks = extract_chunks(SAMPLE_TRANSCRIPT, extraction_output, context_lines=2)

# Show each chunk — field, matched line numbers, and the lines found
for chunk in chunks:
    print(f"Field: {chunk.field}")
    print(f"Lines: {chunk.line_numbers}")
    print("Content:")
    for line in chunk.lines:
        print(f"  {line}")
    print()

In [ ]:
# Show the formatted string that would be injected into the judge prompt
formatted = format_chunks_for_prompt(chunks)
print(formatted)

## Example — chunk search priority (quote → reasoning → fallback)

In [ ]:
# Priority 1: quote field present → should match line 2 (the double-charge line)
chunks_with_quote = extract_chunks(SAMPLE_TRANSCRIPT, extraction_output, context_lines=1)
call_reason_chunk = next(c for c in chunks_with_quote if c.field == 'call_reason')
assert any("charged twice" in line for line in call_reason_chunk.lines), "Quote search should find the double-charge line"
print("PASSED: quote priority — found:", call_reason_chunk.lines)

# Priority 2: no quote → falls back to reasoning phrases
no_quote_output = {k: v for k, v in extraction_output.items() if k != 'quote'}
chunks_no_quote = extract_chunks(SAMPLE_TRANSCRIPT, no_quote_output, context_lines=1)
call_reason_chunk_2 = next(c for c in chunks_no_quote if c.field == 'call_reason')
assert call_reason_chunk_2.lines, "Reasoning fallback should still return lines"
print("PASSED: reasoning fallback — found:", call_reason_chunk_2.lines)

# Priority 3: no quote, no reasoning → falls back to field value
value_only_output = {"call_reason": "cancel subscription"}
chunks_value = extract_chunks(SAMPLE_TRANSCRIPT, value_only_output, context_lines=1)
assert chunks_value[0].lines, "Value fallback should still return lines"
print("PASSED: value fallback — found:", chunks_value[0].lines)

## Mock Unit Test — process_batch with transcript chunker

In [2]:
import json
from unittest.mock import patch
from pydantic import BaseModel
from typing import Literal
from spectrum_client import SpectrumClient
from tools.transcript_chunker import extract_chunks, format_chunks_for_prompt

# --- Pydantic response model the batch expects back from the LLM ---
class CallAnalysis(BaseModel):
    call_reason: str
    sentiment: Literal['positive', 'negative', 'neutral']
    quote: str
    reasoning: str

# --- Two transcript rows to process ---
rows = [
    {
        "AGENTRECORDINGSESSIONID": "SESSION_001",
        "TRANSCRIPT": (
            "Agent: Thank you for calling, how can I help?\n"
            "Customer: I want to cancel my subscription. I've been charged twice this month.\n"
            "Agent: I can see the duplicate charge. I'll process a refund right away.\n"
            "Customer: Thank you."
        ),
    },
    {
        "AGENTRECORDINGSESSIONID": "SESSION_002",
        "TRANSCRIPT": (
            "Agent: Hello, how can I assist you today?\n"
            "Customer: My internet has been down for two days. I need it fixed urgently.\n"
            "Agent: I apologise for the inconvenience. Let me raise a ticket.\n"
            "Customer: Please hurry, I work from home."
        ),
    },
]

# --- Fake LLM responses — one per row (process_batch calls ask_transcript_questions once per row) ---
def fake_response(text):
    return {"output": [{"type": "message", "content": [{"type": "text", "text": text}]}]}

row1_llm_output = json.dumps({
    "call_reason": "billing dispute",
    "sentiment": "negative",
    "quote": "I've been charged twice this month",
    "reasoning": "Customer explicitly mentions a double charge and requests cancellation",
})

row2_llm_output = json.dumps({
    "call_reason": "service outage",
    "sentiment": "negative",
    "quote": "My internet has been down for two days",
    "reasoning": "Customer reports a two-day outage and urgency due to working from home",
})

client = SpectrumClient(url='http://fake', api_key='fake', assistant_id='fake')

with patch.object(client, '_post', side_effect=[fake_response(row1_llm_output), fake_response(row2_llm_output)]):
    results = client.process_batch(
        transcript_column_name='TRANSCRIPT',
        rows=rows,
        prompt_template='Analyse this transcript: {{TRANSCRIPT}}',
        response_format=CallAnalysis,
        template_params={},
        max_workers=1,
    )

assert len(results) == 2, f"Expected 2 results, got {len(results)}"
assert results[0]['AGENTRECORDINGSESSIONID'] == 'SESSION_001'
assert results[0]['PROCESS_STATUS'] == 'TRUE'
assert results[0]['call_reason'] == 'billing dispute'
assert results[1]['AGENTRECORDINGSESSIONID'] == 'SESSION_002'
assert results[1]['call_reason'] == 'service outage'
print('PASSED: process_batch returned correct results for both rows')
print(results)

Processing UCID: SESSION_001

=== RAW MODEL JSON SCHEMA ===
{
  "properties": {
    "call_reason": {
      "title": "Call Reason",
      "type": "string"
    },
    "sentiment": {
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "title": "Sentiment",
      "type": "string"
    },
    "quote": {
      "title": "Quote",
      "type": "string"
    },
    "reasoning": {
      "title": "Reasoning",
      "type": "string"
    }
  },
  "required": [
    "call_reason",
    "sentiment",
    "quote",
    "reasoning"
  ],
  "title": "CallAnalysis",
  "type": "object",
  "additionalProperties": false
}

=== STRICT JSON SCHEMA SENT TO API ===
{
  "properties": {
    "call_reason": {
      "title": "Call Reason",
      "type": "string"
    },
    "sentiment": {
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "title": "Sentiment",
      "type": "string"
    },
    "quote": {
      "title": "Quote",
      "type": "string"

## Mock Unit Test — process_batch failure row (LLM returns None)

In [ ]:
# Simulate a row where the LLM returns an empty response on every attempt
# process_batch should mark it PROCESS_STATUS=FALSE and not raise
empty_response = fake_response('')

# 3 retries per row × 1 row = 3 _post calls
with patch.object(client, '_post', side_effect=[empty_response] * 3):
    fail_results = client.process_batch(
        transcript_column_name='TRANSCRIPT',
        rows=[rows[0]],
        prompt_template='Analyse this transcript: {{TRANSCRIPT}}',
        response_format=CallAnalysis,
        template_params={},
        max_workers=1,
    )

assert fail_results[0]['PROCESS_STATUS'] == 'FALSE'
print('PASSED: failed row correctly marked PROCESS_STATUS=FALSE')

Processing UCID: SESSION_001

=== RAW MODEL JSON SCHEMA ===
{
  "properties": {
    "call_reason": {
      "title": "Call Reason",
      "type": "string"
    },
    "sentiment": {
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "title": "Sentiment",
      "type": "string"
    },
    "quote": {
      "title": "Quote",
      "type": "string"
    },
    "reasoning": {
      "title": "Reasoning",
      "type": "string"
    }
  },
  "required": [
    "call_reason",
    "sentiment",
    "quote",
    "reasoning"
  ],
  "title": "CallAnalysis",
  "type": "object",
  "additionalProperties": false
}

=== STRICT JSON SCHEMA SENT TO API ===
{
  "properties": {
    "call_reason": {
      "title": "Call Reason",
      "type": "string"
    },
    "sentiment": {
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "title": "Sentiment",
      "type": "string"
    },
    "quote": {
      "title": "Quote",
      "type": "string"

# run_extraction_agent + search_transcript Testing

## Example — build_search_tool

In [8]:
from tools.search_transcript import build_search_tool, build_tool_schema
from tools.transcript_chunker import _search_lines

TRANSCRIPT = (
    'Agent: Thank you for calling, how can I help?\n'
    'Customer: I want to cancel my subscription. I have been charged twice this month.\n'
    'Agent: I can see the duplicate charge. I will process a refund right away.\n'
    'Customer: Thank you very much.'
)

search_fn, schema = build_search_tool(TRANSCRIPT)

# hit — should return the line containing 'charged twice' plus context
hit = search_fn(query='charged twice', context_lines=1)
print('HIT:')
print(hit)
assert 'charged twice' in hit.lower()
print('PASSED: hit returns matching lines')

# miss — should return the no-match sentinel
miss = search_fn(query='something that does not exist in the transcript')
assert miss == 'No match found.'
print('PASSED: miss returns No match found.')

# schema shape — must be an OpenAI-style function tool dict
assert schema['type'] == 'function'
assert schema['function']['name'] == 'search_transcript'
assert 'query' in schema['function']['parameters']['properties']
print('PASSED: schema has correct shape')

HIT:
[line 1] Agent: Thank you for calling, how can I help?
[line 2] Customer: I want to cancel my subscription. I have been charged twice this month.
[line 3] Agent: I can see the duplicate charge. I will process a refund right away.
PASSED: hit returns matching lines
PASSED: miss returns No match found.
PASSED: schema has correct shape


## Mock Unit Test — run_extraction_agent inline mode (short transcript)

In [5]:
import json
from unittest.mock import patch
from pydantic import BaseModel
from typing import Literal
from spectrum_client import SpectrumClient

class CallAnalysis(BaseModel):
    call_reason: str
    sentiment: Literal['positive', 'negative', 'neutral']

def fake_response(text):
    return {'output': [{'type': 'message', 'content': [{'type': 'text', 'text': text}]}]}

client = SpectrumClient(url='http://fake', api_key='fake', assistant_id='fake')

# short transcript — token count will be well under 500 so inline mode is used
# expect exactly 1 _post call with the transcript injected into the prompt
short_transcript = 'Agent: Hi. Customer: I want to cancel.'

llm_output = json.dumps({'call_reason': 'cancellation', 'sentiment': 'negative'})

with patch.object(client, '_post', return_value=fake_response(llm_output)) as mock_post:
    result, search_terms = client.run_extraction_agent(
        transcript=short_transcript,
        system_prompt='Extract call info.',
        user_prompt_template='Analyse: {{TRANSCRIPT}}',
        response_format=CallAnalysis,
        token_threshold=500,
    )

assert mock_post.call_count == 1, f'Expected 1 _post call, got {mock_post.call_count}'
# confirm transcript was injected into the prompt payload
payload_sent = mock_post.call_args[0][0]
assert short_transcript in payload_sent['input']
assert result.call_reason == 'cancellation'
assert result.sentiment == 'negative'
assert search_terms == [], f'Inline mode should have no search terms, got {search_terms}'
print('PASSED: inline mode — 1 _post call, transcript in prompt, result validated, no search terms')

False
PASSED: inline mode — 1 _post call, transcript in prompt, result validated, no search terms


## Mock Unit Test — run_extraction_agent tool mode (long transcript)

In [4]:
import json
from unittest.mock import patch
from pydantic import BaseModel
from typing import Literal
from spectrum_client import SpectrumClient

class CallAnalysis(BaseModel):
    call_reason: str
    sentiment: Literal['positive', 'negative', 'neutral']

def fake_response(text):
    return {'output': [{'type': 'message', 'content': [{'type': 'text', 'text': text}]}]}

client = SpectrumClient(url='http://fake', api_key='fake', assistant_id='fake')


# long transcript — force tool mode by setting token_threshold=0
# _post call 1: LLM returns a search_transcript tool call
# _post call 2: LLM returns the final extraction after seeing the tool result

long_transcript = '\n'.join([
    'Agent: Thank you for calling, how can I help?',
    'Customer: I want to cancel my subscription.',
    'Agent: I am sorry to hear that. Can I ask why?',
    'Customer: I have been charged twice this month and I am very frustrated.',
    'Agent: I can see the duplicate charge. I will process a refund right away.',
    'Customer: Thank you, I appreciate that.',
])

# first response: LLM calls the search tool
tool_call_response = {
    'output': [{
        'type': 'function_call',
        'name': 'search_transcript',
        'arguments': json.dumps({'query': 'charged twice', 'context_lines': 2}),
        'call_id': 'call_001',
    }]
}

# second response: LLM returns final answer after seeing the tool result
final_output = json.dumps({'call_reason': 'billing dispute', 'sentiment': 'negative'})

with patch.object(client, '_post', side_effect=[tool_call_response, fake_response(final_output)]) as mock_post:
    result, search_terms = client.run_extraction_agent(
        transcript=long_transcript,
        system_prompt='Extract call info.',
        user_prompt_template='Analyse the transcript using the search tool.',
        response_format=CallAnalysis,
        token_threshold=0,  # force tool mode regardless of length
    )

assert mock_post.call_count == 2, f'Expected 2 _post calls, got {mock_post.call_count}'
# confirm transcript was NOT injected into the first prompt payload
first_payload = mock_post.call_args_list[0][0][0]
assert long_transcript not in first_payload['input']
# confirm the tool schema was included in the first payload
assert any(t['function']['name'] == 'search_transcript' for t in first_payload['tools'])
assert result.call_reason == 'billing dispute'
assert search_terms == ['charged twice'], f'Expected [charged twice], got {search_terms}'
print('PASSED: tool mode — 2 _post calls, transcript not in prompt, tool schema present, result validated')
print('Search terms used:', search_terms)

PASSED: tool mode — 2 _post calls, transcript not in prompt, tool schema present, result validated
Search terms used: ['charged twice']


## Mock Unit Test — process_batch with use_extraction_agent=True

In [11]:
import json
from unittest.mock import patch
from pydantic import BaseModel
from typing import Literal
from spectrum_client import SpectrumClient

class CallAnalysis(BaseModel):
    call_reason: str
    sentiment: Literal['positive', 'negative', 'neutral']

def fake_response(text):
    return {'output': [{'type': 'message', 'content': [{'type': 'text', 'text': text}]}]}

def fake_tool_call(query, call_id):
    return {
        'output': [{
            'type': 'function_call',
            'name': 'search_transcript',
            'arguments': json.dumps({'query': query, 'context_lines': 2}),
            'call_id': call_id,
            
        }]
    }

client = SpectrumClient(url='http://fake', api_key='fake', assistant_id='fake')

rows = [
    {'AGENTRECORDINGSESSIONID': 'S001', 'TRANSCRIPT': 'Agent: Hi. Customer: Cancel please.'},
    {'AGENTRECORDINGSESSIONID': 'S002', 'TRANSCRIPT': 'Agent: Hi. Customer: Billing issue.'},
]

r1_final = json.dumps({'call_reason': 'cancellation', 'sentiment': 'negative'})
r2_final = json.dumps({'call_reason': 'billing dispute', 'sentiment': 'negative'})

mock_side_effects = [
    fake_tool_call('cancel', 'call_001'),
    fake_response(r1_final),
    fake_tool_call('billing', 'call_002'),
    fake_response(r2_final),
]

with patch.object(client, '_post', side_effect=mock_side_effects):
    results = client.process_batch(
        transcript_column_name='TRANSCRIPT',
        rows=rows,
        prompt_template='Analyse transcript using the search tool.',
        response_format=CallAnalysis,
        template_params={},
        system_prompt='Use the search_transcript tool before answering.',
        max_workers=1,
        use_extraction_agent=True,
        token_threshold=0,
    )

print(results)

assert len(results) == 2
assert results[0]['PROCESS_STATUS'] == 'TRUE'
assert results[0]['call_reason'] == 'cancellation'
assert results[1]['call_reason'] == 'billing dispute'

# tool mode: both rows should have the search term the LLM used
assert results[0]['SEARCH_TERMS'] == 'cancel', f"Row 1 search terms: {results[0]['SEARCH_TERMS']}"
assert results[1]['SEARCH_TERMS'] == 'billing', f"Row 2 search terms: {results[1]['SEARCH_TERMS']}"
print('PASSED: process_batch use_extraction_agent=True — both rows extracted correctly with search terms')
print(results)

Processing UCID: S001
[INFO] RAW RESULT: {'output': [{'type': 'function_call', 'name': 'search_transcript', 'arguments': '{"query": "cancel", "context_lines": 2}', 'call_id': 'call_001'}]}
[INFO]: GETTING TOOLS: [{'type': 'function_call', 'name': 'search_transcript', 'arguments': '{"query": "cancel", "context_lines": 2}', 'call_id': 'call_001'}]
[INFO] RAW RESULT: {'output': [{'type': 'message', 'content': [{'type': 'text', 'text': '{"call_reason": "cancellation", "sentiment": "negative"}'}]}]}
[INFO]: GETTING TOOLS: []
['cancel']
{'AGENTRECORDINGSESSIONID': 'S001', 'PROCESS_STATUS': 'TRUE', 'call_reason': 'cancellation', 'sentiment': 'negative', 'CHUNK_CONTEXT': '[call_reason] (lines ):\nAgent: Hi. Customer: Cancel please.\n\n[sentiment] (lines ):\nAgent: Hi. Customer: Cancel please.', 'SEARCH_TERMS': 'cancel'}
Completed 1/2: AGENTRECORDINGSESSIONID S001 | Status: TRUE
Processing UCID: S002
[INFO] RAW RESULT: {'output': [{'type': 'function_call', 'name': 'search_transcript', 'argument

## Mock Unit Test — process_batch use_extraction_agent mixed inline + tool mode

In [13]:
# Ties together process_batch → run_extraction_agent → inline branch (row 1)
# and tool branch (row 2) in a single batch call.
#
# Row 1: short transcript — token count < threshold → inline mode → 1 _post call
# Row 2: token_threshold=0 forces tool mode → 2 _post calls (tool call + final answer)
# Total _post calls: 3

import json
from unittest.mock import patch
from pydantic import BaseModel
from typing import Literal
from spectrum_client import SpectrumClient

class CallAnalysis(BaseModel):
    call_reason: str
    sentiment: Literal['positive', 'negative', 'neutral']

def fake_response(text):
    return {'output': [{'type': 'message', 'content': [{'type': 'text', 'text': text}]}]}

client = SpectrumClient(url='http://fake', api_key='fake', assistant_id='fake')

# Row 1 — short, will go inline regardless of token_threshold
# Row 2 — also short, but token_threshold=0 forces tool mode
mixed_rows = [
    {'AGENTRECORDINGSESSIONID': 'INLINE_001', 'TRANSCRIPT': 'Agent: Hi. Customer: Cancel please.'},
    {'AGENTRECORDINGSESSIONID': 'TOOL_002',   'TRANSCRIPT': 'Agent: Hi. Customer: Billing issue.'},
]

# _post call 1 (row 1, inline): LLM returns final answer directly
inline_answer = fake_response(json.dumps({'call_reason': 'cancellation', 'sentiment': 'negative'}))

# _post call 2 (row 2, tool mode): LLM calls search_transcript
tool_call_resp = {
    'output': [{
        'type': 'function_call',
        'name': 'search_transcript',
        'arguments': json.dumps({'query': 'billing', 'context_lines': 2}),
        'call_id': 'call_abc',
    }]
}

# _post call 3 (row 2, tool mode): LLM returns final answer after seeing tool result
tool_final_answer = fake_response(json.dumps({'call_reason': 'billing dispute', 'sentiment': 'negative'}))

# process_batch processes rows sequentially (max_workers=1)
# row 1 uses token_threshold=500 (inline), row 2 uses token_threshold=0 (tool)
# We patch token_threshold per-row by passing it at the batch level;
# to force tool mode only on row 2 we call process_batch twice and merge,
# OR we set threshold=0 for the whole batch and give row 1 a tool-call + final answer.
#
# Simplest approach: set token_threshold=0 for the whole batch so BOTH rows use
# tool mode — 2 _post calls each = 4 total. This validates the full tool path
# end-to-end through process_batch for every row.

tool_call_row1 = {
    'output': [{
        'type': 'function_call',
        'name': 'search_transcript',
        'arguments': json.dumps({'query': 'cancel', 'context_lines': 2}),
        'call_id': 'call_r1',
    }]
}
tool_final_row1 = fake_response(json.dumps({'call_reason': 'cancellation', 'sentiment': 'negative'}))
tool_call_row2 = tool_call_resp
tool_final_row2 = tool_final_answer

# 4 _post calls total: (tool_call, final) × 2 rows
with patch.object(client, '_post', side_effect=[
    tool_call_row1, tool_final_row1,   # row 1: tool mode
    tool_call_row2, tool_final_row2,   # row 2: tool mode
]) as mock_post:
    mixed_results = client.process_batch(
        transcript_column_name='TRANSCRIPT',
        rows=mixed_rows,
        prompt_template='Analyse the transcript using the search tool.',
        response_format=CallAnalysis,
        template_params={},
        system_prompt='Extract call info.',
        max_workers=1,
        use_extraction_agent=True,
        token_threshold=0,  # force tool mode for every row
    )

assert mock_post.call_count == 4, f'Expected 4 _post calls, got {mock_post.call_count}'

assert len(mixed_results) == 2
assert mixed_results[0]['AGENTRECORDINGSESSIONID'] == 'INLINE_001'
assert mixed_results[0]['PROCESS_STATUS'] == 'TRUE'
assert mixed_results[0]['call_reason'] == 'cancellation'
assert mixed_results[1]['AGENTRECORDINGSESSIONID'] == 'TOOL_002'
assert mixed_results[1]['PROCESS_STATUS'] == 'TRUE'
assert mixed_results[1]['call_reason'] == 'billing dispute'

# confirm tool schema was present in the first call of each row
first_call_payload  = mock_post.call_args_list[0][0][0]
third_call_payload  = mock_post.call_args_list[2][0][0]
assert any(t['name'] == 'search_transcript' for t in first_call_payload['tools'])
assert any(t['name'] == 'search_transcript' for t in third_call_payload['tools'])

# confirm chunk context was written into the output for each row
assert 'CHUNK_CONTEXT' in mixed_results[0], 'Row 1 missing CHUNK_CONTEXT'
assert 'CHUNK_CONTEXT' in mixed_results[1], 'Row 2 missing CHUNK_CONTEXT'
assert mixed_results[0]['CHUNK_CONTEXT'], 'Row 1 CHUNK_CONTEXT is empty'
assert mixed_results[1]['CHUNK_CONTEXT'], 'Row 2 CHUNK_CONTEXT is empty'
# spot-check: the cancellation row should reference the cancel line
assert 'cancel' in mixed_results[0]['CHUNK_CONTEXT'].lower(), 'Expected cancel evidence in row 1 chunk context'
# spot-check: the billing row should reference the billing line
assert 'billing' in mixed_results[1]['CHUNK_CONTEXT'].lower(), 'Expected billing evidence in row 2 chunk context'

print('PASSED: process_batch use_extraction_agent tool mode — 4 _post calls, both rows extracted correctly')
print('Row 1 CHUNK_CONTEXT:')
print(mixed_results[0]['CHUNK_CONTEXT'])
print()
print('Row 2 CHUNK_CONTEXT:')
print(mixed_results[1]['CHUNK_CONTEXT'])
print()
print(mixed_results)

Processing UCID: INLINE_001
[INFO] RAW RESULT: {'output': [{'type': 'function_call', 'name': 'search_transcript', 'arguments': '{"query": "cancel", "context_lines": 2}', 'call_id': 'call_r1'}]}
[INFO]: GETTING TOOLS: [{'type': 'function_call', 'name': 'search_transcript', 'arguments': '{"query": "cancel", "context_lines": 2}', 'call_id': 'call_r1'}]
[INFO] RAW RESULT: {'output': [{'type': 'message', 'content': [{'type': 'text', 'text': '{"call_reason": "cancellation", "sentiment": "negative"}'}]}]}
[INFO]: GETTING TOOLS: []
['cancel']
{'AGENTRECORDINGSESSIONID': 'INLINE_001', 'PROCESS_STATUS': 'TRUE', 'call_reason': 'cancellation', 'sentiment': 'negative', 'CHUNK_CONTEXT': '[call_reason] (lines ):\nAgent: Hi. Customer: Cancel please.\n\n[sentiment] (lines ):\nAgent: Hi. Customer: Cancel please.', 'SEARCH_TERMS': 'cancel'}
Completed 1/2: AGENTRECORDINGSESSIONID INLINE_001 | Status: TRUE
Processing UCID: TOOL_002
[INFO] RAW RESULT: {'output': [{'type': 'function_call', 'name': 'search_tr

KeyError: 'name'

## Integration — chunker output fed into process_batch template_params

In [ ]:
# Demonstrates how extract_chunks + format_chunks_for_prompt can be used
# as a callable template_params to inject focused chunks into the prompt
# instead of the full transcript — mirrors how JudgeAgent uses the chunker.

rows_chunker = [
    {
        'AGENTRECORDINGSESSIONID': 'SESSION_001',
        'TRANSCRIPT': (
            'Agent: Thank you for calling, how can I help?\n'
            'Customer: I want to cancel my subscription. I have been charged twice this month.\n'
            'Agent: I can see the duplicate charge. I will process a refund right away.\n'
            'Customer: Thank you.'
        ),
    },
    {
        'AGENTRECORDINGSESSIONID': 'SESSION_002',
        'TRANSCRIPT': (
            'Agent: Hello, how can I assist you today?\n'
            'Customer: My internet has been down for two days. I need it fixed urgently.\n'
            'Agent: I apologise for the inconvenience. Let me raise a ticket.\n'
            'Customer: Please hurry, I work from home.'
        ),
    },
]

# Simulated prior extraction output per row (would come from a first-pass batch run)
prior_extractions = {
    'SESSION_001': {
        'call_reason': 'billing dispute',
        'sentiment': 'negative',
        'quote': "I've been charged twice this month",
        'reasoning': 'Customer mentions double charge and requests cancellation',
    },
    'SESSION_002': {
        'call_reason': 'service outage',
        'sentiment': 'negative',
        'quote': 'My internet has been down for two days',
        'reasoning': 'Customer reports outage and urgency due to working from home',
    },
}

# callable template_params: receives the row dict, returns the params dict
# CHUNK_CONTEXT replaces {{TRANSCRIPT}} with focused lines instead of the full text
def build_params(row):
    session_id = row['AGENTRECORDINGSESSIONID']
    extraction = prior_extractions[session_id]
    chunks = extract_chunks(row['TRANSCRIPT'], extraction, context_lines=2)
    return {'CHUNK_CONTEXT': format_chunks_for_prompt(chunks)}

# Verify the params are built correctly before wiring into process_batch
params_001 = build_params(rows_chunker[0])
print('SESSION_001 chunk context:')
print(params_001['CHUNK_CONTEXT'])
print()

# Confirm the quote line is present in the chunk context
assert "charged twice" in params_001['CHUNK_CONTEXT'], "Quote should appear in chunk context"
print('PASSED: chunk context contains the expected quote line')